# Day 04 — File Handling, Exceptions, Modules & Basic OOP


In [1]:
import tempfile
from pathlib import Path

# A scratch folder for all examples in this notebook
workdir = Path(tempfile.mkdtemp(prefix="day04_"))
print("Working in:", workdir)

Working in: /tmp/day04_2xo653nb


---
## 1. File Handling

The standard pattern is `open(path, mode)` inside a `with` block, which **closes the file automatically**, even if an error occurs.

| Mode | Meaning |
|---|---|
| `"r"` | Read (default). Error if the file doesn't exist |
| `"w"` | Write. Creates the file or **overwrites** it |
| `"a"` | Append. Creates the file or adds to the end |
| `"x"` | Create only. Error if the file already exists |
| `"r+"` | Read and write |
| `"b"` | Add to any mode for binary (`"rb"`, `"wb"`) |

### 1.1 Writing a file

In [2]:
path = workdir / "notes.txt"

with open(path, "w", encoding="utf-8") as f:
    f.write("Line one\n")
    f.write("Line two\n")
    f.writelines(["Line three\n", "Line four\n"])

print("Saved to", path.name)

Saved to notes.txt


### 1.2 Reading a file

In [3]:
# Read everything as one string
with open(path, encoding="utf-8") as f:
    content = f.read()
print(repr(content))

# Read all lines into a list
with open(path, encoding="utf-8") as f:
    lines = f.readlines()
print(lines)

# Read one line at a time (memory-friendly for big files)
with open(path, encoding="utf-8") as f:
    for number, line in enumerate(f, start=1):
        print(number, line.rstrip())

'Line one\nLine two\nLine three\nLine four\n'
['Line one\n', 'Line two\n', 'Line three\n', 'Line four\n']
1 Line one
2 Line two
3 Line three
4 Line four


### 1.3 Appending

In [4]:
with open(path, "a", encoding="utf-8") as f:
    f.write("Line five (appended)\n")

print(path.read_text(encoding="utf-8"))

Line one
Line two
Line three
Line four
Line five (appended)



### 1.4 The file cursor: `seek()` and `tell()`

In [5]:
with open(path, encoding="utf-8") as f:
    print(f.read(8))      # first 8 characters
    print("position:", f.tell())
    f.seek(0)             # jump back to the start
    print(f.readline().rstrip())

Line one
position: 8
Line one


### 1.5 `pathlib` — a modern way to work with paths

In [6]:
p = workdir / "data" / "report.csv"      # build paths with /

print(p.name)        # report.csv
print(p.stem)        # report
print(p.suffix)      # .csv
print(p.parent.name) # data

p.parent.mkdir(exist_ok=True)            # create folder
p.write_text("id,name\n1,Ana\n", encoding="utf-8")

print(p.exists(), p.is_file(), p.parent.is_dir())
print(p.read_text(encoding="utf-8"))

# List files
for item in workdir.iterdir():
    print("-", item.name)

print([x.name for x in workdir.glob("*.txt")])

report.csv
report
.csv
data
True True True
id,name
1,Ana

- data
- notes.txt
['notes.txt']


### 1.6 CSV files

In [7]:
import csv

csv_path = workdir / "students.csv"
rows = [
    {"name": "Ana", "age": 21, "score": 91},
    {"name": "Ben", "age": 22, "score": 84},
    {"name": "Cy",  "age": 20, "score": 77},
]

# Write
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "age", "score"])
    writer.writeheader()
    writer.writerows(rows)

# Read
with open(csv_path, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(row["name"], int(row["score"]))

Ana 91
Ben 84
Cy 77


### 1.7 JSON files

In [8]:
import json

config = {
    "app": "demo",
    "debug": True,
    "tags": ["python", "files"],
    "limits": {"max_users": 100},
}

json_path = workdir / "config.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

with open(json_path, encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded == config)
print(loaded["limits"]["max_users"])

# To/from strings
text = json.dumps(config)
print(type(text), json.loads(text)["app"])

True
100
<class 'str'> demo


### 1.8 Binary files

In [9]:
bin_path = workdir / "bytes.bin"

with open(bin_path, "wb") as f:
    f.write(bytes([72, 101, 108, 108, 111]))

with open(bin_path, "rb") as f:
    data = f.read()

print(data, list(data), data.decode())

b'Hello' [72, 101, 108, 108, 111] Hello


---
## 2. Exceptions

An **exception** is an error raised while the program runs. Handle it with `try / except` so your program can recover instead of crashing.

```
try:        code that might fail
except:     what to do if it fails
else:       runs only if NO exception happened
finally:    always runs (cleanup)
```

### 2.1 Common exceptions

In [11]:
def show(func):
    try:
        func()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

show(lambda: 10 / 0)                   # ZeroDivisionError
show(lambda: int("abc"))               # ValueError
show(lambda: [1, 2, 3][10])            # IndexError
show(lambda: {"a": 1}["b"])            # KeyError
show(lambda: "text" + 5)               # TypeError
show(lambda: undefined_name)           # NameError
show(lambda: (1).missing)              # AttributeError
show(lambda: open("no_such_file.txt")) # FileNotFoundError

ZeroDivisionError: division by zero
ValueError: invalid literal for int() with base 10: 'abc'
IndexError: list index out of range
KeyError: 'b'
TypeError: can only concatenate str (not "int") to str
NameError: name 'undefined_name' is not defined
AttributeError: 'int' object has no attribute 'missing'
FileNotFoundError: [Errno 2] No such file or directory: 'no_such_file.txt'


### 2.2 `try` / `except`

In [12]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

5.0
Cannot divide by zero
None


### 2.3 Multiple `except` blocks

In [13]:
def parse_and_divide(a, b):
    try:
        return int(a) / int(b)
    except ValueError:
        return "Please enter whole numbers"
    except ZeroDivisionError:
        return "Second number can't be zero"

print(parse_and_divide("10", "4"))
print(parse_and_divide("ten", "4"))
print(parse_and_divide("10", "0"))

# Catch several types in one block
try:
    [1, 2][5]
except (IndexError, KeyError) as e:
    print("Lookup problem:", e)

2.5
Please enter whole numbers
Second number can't be zero
Lookup problem: list index out of range


### 2.4 `else` and `finally`

In [14]:
def read_number_file(p):
    try:
        f = open(p)
    except FileNotFoundError:
        print("  file not found")
    else:
        print("  opened OK, reading...")
        value = f.read().strip()
        f.close()
        return value
    finally:
        print("  (finally always runs)")

(workdir / "n.txt").write_text("42")
print(read_number_file(workdir / "n.txt"))
print(read_number_file(workdir / "missing.txt"))

  opened OK, reading...
  (finally always runs)
42
  file not found
  (finally always runs)
None


### 2.5 Raising exceptions

In [15]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError("age must be an integer")
    if age < 0:
        raise ValueError("age cannot be negative")
    return age

for value in [25, -3, "ten"]:
    try:
        print("OK:", set_age(value))
    except (TypeError, ValueError) as e:
        print(f"{type(e).__name__}: {e}")

OK: 25
ValueError: age cannot be negative
TypeError: age must be an integer


### 2.6 Custom exceptions

In [16]:
class InsufficientFundsError(Exception):
    '''Raised when a withdrawal exceeds the balance.'''

    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"Cannot withdraw {amount}; balance is only {balance}")

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount

try:
    withdraw(100, 250)
except InsufficientFundsError as e:
    print(e)
    print("Short by:", e.amount - e.balance)

Cannot withdraw 250; balance is only 100
Short by: 150


### 2.7 Re-raising, chaining and `assert`

In [17]:
# Re-raise after logging
def load(path):
    try:
        return open(path).read()
    except FileNotFoundError:
        print("log: file missing")
        raise            # pass the same exception upward

try:
    load(workdir / "nope.txt")
except FileNotFoundError:
    print("handled at a higher level")

# Chain exceptions with 'from' to keep the original cause
try:
    try:
        int("x")
    except ValueError as e:
        raise RuntimeError("config is invalid") from e
except RuntimeError as e:
    print(e, "| caused by:", repr(e.__cause__))

# assert: quick sanity check during development
try:
    assert 2 + 2 == 5, "math is broken"
except AssertionError as e:
    print("AssertionError:", e)

log: file missing
handled at a higher level
config is invalid | caused by: ValueError("invalid literal for int() with base 10: 'x'")
AssertionError: math is broken


### Best practices
- Catch **specific** exceptions, not a bare `except:`.
- Keep the `try` block small — only the code that can fail.
- Use `finally` or `with` for cleanup.
- Don't silence errors with `pass` unless you truly mean to.


---
## 3. Modules

A **module** is a `.py` file containing code you can reuse. A **package** is a folder of modules. Python's standard library ships with hundreds.

### 3.1 Importing

In [19]:
import math                          # whole module
from math import sqrt, pi            # specific names
import random as rnd                 # alias
from datetime import datetime, timedelta

print(math.floor(3.7), math.ceil(3.2))
print(sqrt(81), round(pi, 4))
print(rnd.randint(1, 6) in range(1, 7))

3 4
9.0 3.1416
True


### 3.2 Handy standard-library modules

In [20]:
import math, random, datetime, os, sys, statistics, collections

# math
print(math.factorial(5), math.gcd(12, 18), math.isclose(0.1 + 0.2, 0.3))

# random (seeded so results repeat)
random.seed(42)
print(random.choice(["a", "b", "c"]), random.sample(range(10), 3))

# datetime
now = datetime.datetime(2026, 10, 16, 9, 30)
print(now.strftime("%A, %d %B %Y"))
print(now + datetime.timedelta(days=30))

# statistics
scores = [90, 85, 77, 92, 85]
print(statistics.mean(scores), statistics.median(scores), statistics.mode(scores))

# os / sys
print(os.path.basename("/a/b/c.txt"), sys.version_info[:2])

# collections
print(collections.Counter("banana"))

120 6 True
c [1, 0, 4]
Friday, 16 October 2026
2026-11-15 09:30:00
85.8 85 85
c.txt (3, 13)
Counter({'a': 3, 'n': 2, 'b': 1})


### 3.3 Exploring a module

In [21]:
import math

print([n for n in dir(math) if not n.startswith("_")][:10], "...")
print(math.sqrt.__doc__)
print(math.__name__)

['acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'cbrt', 'ceil', 'comb'] ...
Return the square root of x.
math


### 3.4 Creating your own module

In [22]:
module_code = '''
"""A tiny helper module."""

PI = 3.14159

def area_circle(r):
    return PI * r ** 2

def greet(name):
    return f"Hello, {name}!"

if __name__ == "__main__":
    print("Running mytools directly")
'''
(workdir / "mytools.py").write_text(module_code, encoding="utf-8")

import sys
sys.path.insert(0, str(workdir))     # make the folder importable

import mytools

print(mytools.greet("Sara"))
print(round(mytools.area_circle(2), 2))
print(mytools.__name__)              # 'mytools' when imported
print(mytools.__doc__)

Hello, Sara!
12.57
mytools
A tiny helper module.


### 3.5 The `if __name__ == "__main__":` guard

- When a file is **run directly**, `__name__` is `"__main__"`.
- When it's **imported**, `__name__` is the module's name.

The guard lets a file work as both a reusable module and a script.

In [23]:
import subprocess

# Run the module as a script
out = subprocess.run([sys.executable, str(workdir / "mytools.py")],
                     capture_output=True, text=True)
print(out.stdout.strip())

Running mytools directly


### 3.6 Third-party packages

In [24]:
# Install with:  pip install requests   (in a terminal)
# or in a notebook cell:  %pip install requests

import importlib.util
print("numpy installed:", importlib.util.find_spec("numpy") is not None)

# Defensive import for optional dependencies
try:
    import numpy as np
    print("numpy version:", np.__version__)
except ImportError:
    print("numpy not installed, skipping")

numpy installed: True


numpy version: 2.5.3


---
## 4. Basic Object-Oriented Programming

OOP organizes code into **classes** (blueprints) that create **objects** (instances) bundling **data (attributes)** and **behavior (methods)**.

| Term | Meaning |
|---|---|
| Class | Blueprint for objects |
| Object / instance | A concrete thing built from a class |
| Attribute | A variable stored on an object |
| Method | A function defined inside a class |
| `self` | The current instance, passed automatically |

### 4.1 first class

In [26]:
class Dog:
    species = "Canis familiaris"        # class attribute (shared)

    def __init__(self, name, age):      # constructor
        self.name = name                # instance attributes
        self.age = age

    def bark(self):
        return f"{self.name} says Woof!"

    def birthday(self):
        self.age += 1

rex = Dog("Rex", 3)
bella = Dog("Bella", 5)

print(rex.bark())
rex.birthday()
print(rex.name, rex.age)
print(bella.name, bella.age)
print(Dog.species, rex.species)

Rex says Woof!
Rex 4
Bella 5
Canis familiaris Canis familiaris


### 4.2 `__str__` and `__repr__`

In [27]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def __str__(self):                       # friendly, for print()
        return f"'{self.title}' by {self.author}"

    def __repr__(self):                      # unambiguous, for debugging
        return f"Book({self.title!r}, {self.author!r}, {self.pages})"

b = Book("Dune", "Frank Herbert", 412)
print(b)
print(repr(b))
print([b])

'Dune' by Frank Herbert
Book('Dune', 'Frank Herbert', 412)
[Book('Dune', 'Frank Herbert', 412)]


### 4.3 Encapsulation: protecting data

In [28]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance      # leading _ = "internal, please don't touch"

    @property
    def balance(self):               # read-only access
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount

    def withdraw(self, amount):
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount

acct = BankAccount("Ali", 100)
acct.deposit(50)
acct.withdraw(30)
print(acct.owner, acct.balance)

try:
    acct.balance = 1_000_000          # property has no setter
except AttributeError as e:
    print("AttributeError:", e)

Ali 120
AttributeError: property 'balance' of 'BankAccount' object has no setter


### 4.4 Class methods and static methods

In [29]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, f):          # alternative constructor
        return cls((f - 32) * 5 / 9)

    @staticmethod
    def is_freezing(celsius):             # utility, no self/cls needed
        return celsius <= 0

    def __repr__(self):
        return f"Temperature({self.celsius:.1f}°C)"

t = Temperature.from_fahrenheit(212)
print(t)
print(Temperature.is_freezing(-5))

Temperature(100.0°C)
True


### 4.5 Inheritance

In [30]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

    def intro(self):
        return f"I'm {self.name} and I say {self.speak()}"

class Cat(Animal):
    def speak(self):                      # override
        return "Meow"

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)            # reuse parent's __init__
        self.breed = breed

    def speak(self):
        return "Woof"

animals = [Cat("Tom"), Dog("Rex", "Husky"), Animal("Generic")]
for a in animals:
    print(a.intro())

print(isinstance(animals[0], Animal), issubclass(Dog, Animal))

I'm Tom and I say Meow


I'm Rex and I say Woof
I'm Generic and I say ...
True True


### 4.6 Polymorphism

In [31]:
class Shape:
    def area(self):
        raise NotImplementedError("Subclasses must implement area()")

class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h
    def area(self):
        return self.w * self.h

class Circle(Shape):
    def __init__(self, r):
        self.r = r
    def area(self):
        return 3.14159 * self.r ** 2

shapes = [Rectangle(3, 4), Circle(2)]
for s in shapes:                       # same call, different behavior
    print(type(s).__name__, round(s.area(), 2))

print("Total area:", round(sum(s.area() for s in shapes), 2))

Rectangle 12
Circle 12.57
Total area: 24.57


### 4.7 Special (dunder) methods

In [32]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

    def __len__(self):
        return 2

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v1, v2 = Vector(1, 2), Vector(3, 4)
print(v1 + v2)
print(v1 == Vector(1, 2))
print(len(v1))

Vector(4, 6)
True
2


### 4.8 Composition and dataclasses

In [33]:
# Composition: an object HAS other objects ("has-a")
class Engine:
    def start(self):
        return "Engine started"

class Car:
    def __init__(self, model):
        self.model = model
        self.engine = Engine()

    def start(self):
        return f"{self.model}: {self.engine.start()}"

print(Car("Civic").start())

# dataclass: less boilerplate for classes that mostly hold data
from dataclasses import dataclass, field

@dataclass
class Student:
    name: str
    grade: int
    courses: list = field(default_factory=list)

    def enroll(self, course):
        self.courses.append(course)

s = Student("Ana", 10)
s.enroll("Math")
print(s)
print(s == Student("Ana", 10, ["Math"]))

Civic: Engine started
Student(name='Ana', grade=10, courses=['Math'])
True


---
## 5. Putting It All Together: A Mini Task Manager

This combines classes, file handling (JSON), and exception handling.

In [35]:
import json
from dataclasses import dataclass, asdict

class TaskNotFoundError(Exception):
    pass

@dataclass
class Task:
    id: int
    title: str
    done: bool = False

class TaskManager:
    def __init__(self, filepath):
        self.filepath = Path(filepath)
        self.tasks = []
        self._load()

    def _load(self):
        try:
            with open(self.filepath, encoding="utf-8") as f:
                self.tasks = [Task(**t) for t in json.load(f)]
        except FileNotFoundError:
            self.tasks = []                       # first run
        except json.JSONDecodeError:
            print("Corrupt file, starting fresh")
            self.tasks = []

    def _save(self):
        with open(self.filepath, "w", encoding="utf-8") as f:
            json.dump([asdict(t) for t in self.tasks], f, indent=2)

    def add(self, title):
        new_id = max((t.id for t in self.tasks), default=0) + 1
        self.tasks.append(Task(new_id, title))
        self._save()

    def complete(self, task_id):
        for t in self.tasks:
            if t.id == task_id:
                t.done = True
                self._save()
                return
        raise TaskNotFoundError(f"No task with id {task_id}")

    def show(self):
        for t in self.tasks:
            print(f"[{'x' if t.done else ' '}] {t.id}. {t.title}")

db = workdir / "tasks.json"
tm = TaskManager(db)
tm.add("Finish Day 04 notebook")
tm.add("Practice OOP")
tm.complete(1)

try:
    tm.complete(99)
except TaskNotFoundError as e:
    print("Error:", e)

tm.show()

# A new manager reloads the saved data from disk
print("--- reloaded ---")
TaskManager(db).show()

Error: No task with id 99
[x] 1. Finish Day 04 notebook
[ ] 2. Practice OOP
--- reloaded ---
[x] 1. Finish Day 04 notebook
[ ] 2. Practice OOP


In [36]:
# Cleanup: remove the temporary folder
import shutil
shutil.rmtree(workdir, ignore_errors=True)
print("Cleaned up:", not workdir.exists())

Cleaned up: True
